# SCBE Model Training — Pivot Knowledge v2

Trains Qwen2.5-0.5B on the 800 SFT pairs we just generated.
Data source: Google Drive `/SCBE_Training/pivot_sft_v2.jsonl`
Output: `issdandavis/scbe-pivot-qwen-0.5b` on HuggingFace

In [ ]:
!pip install -q "unsloth[colab-new]" trl peft accelerate bitsandbytes datasets huggingface_hub

from google.colab import drive
drive.mount('/content/drive')
print('Ready.')

In [ ]:
from unsloth import FastLanguageModel
import torch, json
from pathlib import Path
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

# Load small base model (fits on T4 in 4bit)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-0.5B-Instruct",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

# Add LoRA adapters (train ~1% of params)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)
print(f'Model loaded. Trainable params: {model.print_trainable_parameters()}')

In [ ]:
# Load our SFT data from Drive
sft_path = Path('/content/drive/MyDrive/SCBE_Training/pivot_sft_v2.jsonl')
rows = [json.loads(line) for line in sft_path.read_text().splitlines() if line.strip()]

# Format as chat messages
def format_chat(row):
    return tokenizer.apply_chat_template([
        {"role": "user", "content": row["instruction"]},
        {"role": "assistant", "content": row["response"]},
    ], tokenize=False)

formatted = [{"text": format_chat(r)} for r in rows]
dataset = Dataset.from_list(formatted)
print(f'Training on {len(dataset)} examples')
print(f'Sample: {formatted[0]["text"][:200]}...')

In [ ]:
# Train!
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=SFTConfig(
        output_dir="/content/scbe_pivot_model",
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        save_strategy="epoch",
        dataset_text_field="text",
        max_seq_length=2048,
    ),
    tokenizer=tokenizer,
)

print('Starting training...')
stats = trainer.train()
print(f'Training complete! Loss: {stats.training_loss:.4f}')

In [ ]:
# Save and push to HuggingFace
model.save_pretrained("/content/scbe_pivot_model")
tokenizer.save_pretrained("/content/scbe_pivot_model")

from huggingface_hub import HfApi
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')
except:
    hf_token = input('Enter HuggingFace token: ')

api = HfApi(token=hf_token)
api.upload_folder(
    folder_path="/content/scbe_pivot_model",
    repo_id="issdandavis/scbe-pivot-qwen-0.5b",
    repo_type="model",
    create_pr=False,
)
print('Model pushed to https://huggingface.co/issdandavis/scbe-pivot-qwen-0.5b')

In [ ]:
# Quick test — ask it something
FastLanguageModel.for_inference(model)

test_questions = [
    'What are the six Sacred Tongues?',
    'Why does SCBE use hyperbolic geometry?',
    'Who is Polly in the Spiralverse?',
]

for q in test_questions:
    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": q}],
        tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to('cuda')
    outputs = model.generate(inputs, max_new_tokens=200, temperature=0.7)
    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    print(f'Q: {q}')
    print(f'A: {response}')
    print()